# 04 — SparkSession

**Module 4 of 15**

## Learning Objectives
- Understand what `SparkSession` is and why it replaced `SparkContext`/`SQLContext`
- Create a `SparkSession` with common configuration options
- Understand the relationship between `SparkSession`, `SparkContext`, and `SQLContext`
- Read the Spark UI to inspect a running application
- Properly stop a `SparkSession`

---

> **A note on running these cells:** From this notebook onward, the code cells assume PySpark and Java are installed in your kernel's environment (see Notebook 03). Run these notebooks in your local Jupyter/venv, Databricks Community Edition, or Google Colab.


## 4.1 What Is a SparkSession?

`SparkSession` is the single, unified entry point for all Spark functionality since Spark 2.0. Before that, you needed separate objects:

- `SparkContext` — the entry point for RDD-based operations (Spark Core)
- `SQLContext` — the entry point for DataFrame/SQL operations
- `HiveContext` — for Hive integration

`SparkSession` wraps all of these into one object, simplifying your code. You'll still see `SparkContext` referenced (e.g., `spark.sparkContext`) because it's used internally and for low-level RDD work (Notebook 05).


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Module04_SparkSession")
    .master("local[*]")          # use all available local cores
    .config("spark.sql.shuffle.partitions", "8")  # smaller default for local learning (default is 200)
    .getOrCreate()
)

spark


## 4.2 Anatomy of `SparkSession.builder`

```python
SparkSession.builder
    .appName("MyApp")            # name shown in the Spark UI
    .master("local[*]")          # cluster manager + resources: local[*] uses all CPU cores
    .config("key", "value")      # any Spark configuration property
    .enableHiveSupport()         # optional: enables Hive metastore integration
    .getOrCreate()               # returns existing session if one exists, else creates a new one
```

### Common `master()` values

| Value | Meaning |
|---|---|
| `local` | Single thread, no parallelism |
| `local[4]` | 4 threads locally |
| `local[*]` | Use all available CPU cores locally |
| `yarn` | Submit to a YARN cluster |
| `spark://host:port` | Submit to a Spark Standalone cluster |
| `k8s://https://host:port` | Submit to a Kubernetes cluster |

### `getOrCreate()` matters

Calling `SparkSession.builder...getOrCreate()` multiple times **reuses the existing session** instead of creating a new JVM each time — this is important in notebooks, where you might re-run cells repeatedly. Only `spark.stop()` truly tears it down.


In [ ]:
# Inspect key attributes of the session
print("App name:      ", spark.sparkContext.appName)
print("Master:        ", spark.sparkContext.master)
print("Spark version: ", spark.version)
print("Default parallelism:", spark.sparkContext.defaultParallelism)


## 4.3 Useful Configuration Options

You can set configuration either at creation time (`.config(...)`) or afterward via `spark.conf.set(...)` for *runtime-modifiable* settings (not all settings can be changed after the session starts).


In [ ]:
# Reading and setting runtime configs
print("Current shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

spark.conf.set("spark.sql.shuffle.partitions", "4")
print("Updated shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

# Some other commonly tuned configs (we'll revisit these in Notebook 11 - Optimization):
# spark.conf.set("spark.executor.memory", "4g")
# spark.conf.set("spark.sql.adaptive.enabled", "true")   # Adaptive Query Execution (on by default since Spark 3.2)


## 4.4 The Spark UI

Every running `SparkSession` launches a web UI (default port `4040`, or `4041`, `4042`... if that port is taken) where you can inspect:

- **Jobs** — every action you've triggered, with timing
- **Stages** — the shuffle boundaries within each job
- **Storage** — cached DataFrames/RDDs and how much memory they use
- **Environment** — every active configuration property
- **Executors** — resource usage per executor

While running locally, visit `http://localhost:4040` in your browser while the SparkSession is alive. This is an invaluable debugging tool once you get into performance tuning (Notebook 11).


In [ ]:
print("If running locally, open this URL while the session is active:")
print(spark.sparkContext.uiWebUrl)


## 4.5 Stopping a SparkSession

Always stop your session when you're done, especially in scripts (in long-lived notebooks it's common to keep one session alive for the whole workflow):

```python
spark.stop()
```

Forgetting to stop a session in a script can leave orphaned JVM processes holding onto memory and ports.


In [ ]:
# We'll keep this session alive for now since later cells in a real run might reuse it,
# but here's how you would cleanly stop it:

# spark.stop()
print("Session left running for the rest of this notebook's examples.")


## 4.6 Summary

- `SparkSession` is the single unified entry point to Spark since 2.0 (replacing `SparkContext` + `SQLContext`).
- Configure it with `.appName()`, `.master()`, `.config()`.
- `getOrCreate()` avoids creating duplicate sessions inside a notebook.
- The Spark UI (`localhost:4040` by default) is essential for debugging and performance analysis.
- Always call `spark.stop()` when a script/session is truly finished.

## Check Your Understanding
1. What three legacy entry points did `SparkSession` unify?
2. What does `master("local[*]")` mean?
3. Where would you go to see how long each stage of your job took?
4. Why is `getOrCreate()` important inside a notebook environment?

---
**Next:** [05_RDD_Basics.ipynb](05_RDD_Basics.ipynb) — Spark's original, low-level distributed data structure.
